# Surface pass: cached activations -> surface, coordinates and CSVs

The CPU half of the arc-length pipeline. For every model it finds under `ARTIFACT_ROOT` it fits the BCPC target, the weighted PLS, the rotated plane and the height-slice surface from the cached fitting activations, exports and plots that surface, projects every cached inference corpus through it, and parses the cached rating continuations into their CSVs.

Nothing here loads model weights or touches a GPU: `notebooks/arc_length_cache.ipynb` did that, and every stage below reads `.pt` and `.json` caches on the CPU. Each model describes itself through the `run_config.json` the caching pass wrote, so the layer, position, batch size and stakes merges never have to be retyped or kept in sync by hand.

Re-running this notebook is cheap and repeatable. Changing a merge rule, a slice setting or a rating parse rule means re-running it, not re-running the GPU pass.

In [ ]:
from pathlib import Path
import sys
import traceback

from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'stakes_surface_pipeline.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import inference_datasets, stated_stakes
from scripts.pipeline_config import discover_run_dirs, load_run_config
from scripts.stakes_surface_pipeline import StakesSurfacePipeline

print('Repository:', ROOT)
print(f'{len(inference_datasets.DATASETS)} inference corpora:',
      ', '.join(dataset.name for dataset in inference_datasets.DATASETS))

## Configuration

`ARTIFACT_ROOT` is the tree the caching pass wrote, whether it was produced here or copied from a GPU host. `MODELS` is `None` to analyse every model directory in it that carries a `run_config.json`, or a list of directory names to restrict the run to some of them.

`RATING_DATASETS` selects which corpora's stated-stakes ratings are exported; `None` means every corpus the caching pass actually generated ratings for. An inference corpus with no cached activations is reported and skipped rather than failing the model, which is what happens when a corpus is added after a model was cached, or when only part of an artifact tree was copied across.

In [ ]:
ARTIFACT_ROOT = ROOT / 'artifacts'
MODELS = None                 # None = every model directory with a run_config.json
RATING_DATASETS = None        # None = every corpus whose ratings were generated

## Analysis helpers

`analyse_model` runs one model end to end: fit, export, plot, project each cached inference corpus, then export the ratings. It rebuilds that model's `RunConfig` from disk, so the settings it uses are the ones its caches were made with.

In [ ]:
def show_inference_result(label, result):
    csv_path, rows, diagnostics = result
    display(rows)
    display(diagnostics.groupby('coordinate_status').size().rename('rows').to_frame())
    display(diagnostics[['outside_saved_height_range', 'surface_extended', 'height_extrapolated']].sum().rename('rows').to_frame())
    display(diagnostics[['surface_projection_residual', 'slice_height_error']].describe())
    print(f'{label} CSV:', csv_path)


def show_rating_result(label, result):
    csv_path, summary_path, table, summary = result
    display(stated_stakes.parse_rate(table))
    display(summary[['ratings_parsed', 'rating_median', 'rating_spread']].describe())
    display(table.head())
    print(f'{label} ratings CSV:', csv_path)
    print(f'{label} per-prompt summary:', summary_path)


def analyse_model(run_dir):
    """Fit and export one model's surface, then every readout that hangs off it."""
    config = load_run_config(run_dir)
    print()
    print(f'=== {config.model_name}: {config.layer_component} ===')
    print(config.describe())

    pipeline = StakesSurfacePipeline(config)
    display(pipeline.load_caches())
    projected_rows = pipeline.fit_bcpc()
    bcpc = pipeline.bcpc
    print(f'{len(projected_rows):,} rows, {len(bcpc.projection["classes"])} merged classes')
    display(bcpc.class_table()); display(bcpc.variance_table())
    display(bcpc.centroids); display(bcpc.anchors)
    print(f'Weighted squared residual sum: {bcpc.weighted_residual_sum:.6g}')
    display(bcpc.spline_points)
    print(f's=0: spline endpoint associated with {bcpc.zero_anchor}')
    print(f'Total spline arc length: {bcpc.total_arc_length:.6g}')
    display(projected_rows[['stakes', 'spline_parameter', 'bcpc_arc_length', 'distance_to_spline']].head())
    display(pipeline.fit_pls()); display(pipeline.weight_summary)
    rotation_diagnostics, file_centroids = pipeline.rotate_plane()
    display(rotation_diagnostics); display(file_centroids)
    display(pipeline.fit_surface())
    display(pipeline.project_stated_rows())
    display(pipeline.build_slice_cache())
    mapping = pipeline.map_coordinates()
    for table in mapping.values():
        display(table)
    display(pipeline.all_rows[['source_file', 'stakes', 'bcpc_arc_length',
                               'arc_length_parallel', 'arc_length_orthogonal']].head())
    output_dir = pipeline.export(notebook='notebooks/arc_length_surface.ipynb')
    figures = pipeline.save_plots()
    for figure in figures.values():
        figure.show()

    available = inference_datasets.cached_datasets(config)
    skipped = [dataset.name for dataset in inference_datasets.DATASETS
               if dataset not in available]
    if skipped:
        print('No cached activations yet for:', ', '.join(skipped))
    for dataset in available:
        show_inference_result(dataset.label, dataset.project(config))

    # Behavioural cross-check: the model's own stated stakes for the same prompts,
    # parsed from the continuations the caching pass generated.
    rated = stated_stakes.cached_datasets(config, RATING_DATASETS)
    if not rated:
        print('No cached rating continuations for this model.')
    for name, result in stated_stakes.export_all(config, rated).items():
        show_rating_result(name, result)
    return output_dir

## Analyse every cached model

A failed model does not prevent later ones from running. Its traceback is recorded and the cell raises after printing the complete status table.

In [ ]:
run_dirs = discover_run_dirs(ARTIFACT_ROOT)
if MODELS is not None:
    wanted = list(MODELS)
    missing = [name for name in wanted if ARTIFACT_ROOT / name not in run_dirs]
    if missing:
        raise ValueError(f'No cached run for {missing} under {ARTIFACT_ROOT}; '
                         f'available: {[path.name for path in run_dirs]}')
    run_dirs = [ARTIFACT_ROOT / name for name in wanted]
if not run_dirs:
    raise ValueError(f'No model under {ARTIFACT_ROOT} has a run_config.json; '
                     'run notebooks/arc_length_cache.ipynb first.')
print(f'{len(run_dirs)} model(s):', ', '.join(path.name for path in run_dirs))

run_results = {}
for run_dir in run_dirs:
    try:
        output_dir = analyse_model(run_dir)
        run_results[run_dir.name] = {'status': 'complete', 'output_dir': str(output_dir)}
    except Exception as exc:
        run_results[run_dir.name] = {'status': 'failed',
                                     'error': f'{type(exc).__name__}: {exc}',
                                     'traceback': traceback.format_exc()}
        print(run_results[run_dir.name]['traceback'])

display(run_results)
failures = {name: result for name, result in run_results.items() if result['status'] != 'complete'}
if failures:
    raise RuntimeError(f'Analysis failures: {list(failures)}')

## Reusing an exported surface

```python
from scripts.stakes_surface_bundle import load_surface_bundle, project_saved_bcpc
arrays, metadata, saved_coordinates = load_surface_bundle(config.surface_dir)
bcpc_projection = project_saved_bcpc(activation_batch, arrays, metadata)
pls_scores = (activation_batch - arrays['pls_mean']) @ arrays['pls_rotations']
new_surface_coordinates = saved_coordinates.map_points(pls_scores[:, :3])
```